# CS604 — look at images before choosing training settings
This notebook helps you see which crops, sizes, zoom levels and small appearance changes are worth trying in training. It **does not train a model, load a checkpoint, change images, or create a holdout**. There are no manifest audits or duplicate scans.

Run the cells in order. A CPU runtime is enough. Images stay on Colab's temporary disk; pictures and tables are saved to Drive. The setup follows `colab_launcher_qq.ipynb`, with separate cells so you can see which step is taking time.

The pictures help you shortlist settings. They do not prove a setting improves accuracy. Official validation images have no labels, so their class accuracy cannot be measured here.


In [ ]:
# Change these settings before running.
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/dataset-inspection-repo"  # Separate from your training checkout.
BRANCH = "dataset-inspection"  # No merge with main is needed.
DATA_DIR = "/content/data"
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
CONFIG = "configs/bitfit_cosine_448.yaml"  # Current training transform to display.
DOWNLOAD_DATA = True  # False if extracted images already exist at DATA_DIR.
SEED = 42
SAMPLE_PER_SPLIT = 1000  # Limits inspection, NOT the dataset download.
CLASS_IDS = []  # Empty picks four training classes; otherwise enter numeric IDs.
EXAMPLE_PATHS = []  # Optional paths from train.txt / val.txt, relative to DATA_DIR.
EXAMPLE_COUNT = 6
RESOLUTIONS = [224, 280, 336, 384, 448, 512]  # ViT-B/16: use multiples of 16.
COMPARE_SIZE = 336  # Fixed output size for crop, zoom, flip and color comparisons.
ZOOM_KEEP = [1.0, 0.85, 0.65, 0.45]  # Fraction of the square view's side to keep.
ZOOM_CENTER = (0.5, 0.5)  # (x, y): 0 = left/top, 1 = right/bottom.
# Alternatives for random training crops: fraction of image AREA to keep.
CROP_RANGES = [(0.08, 1.0), (0.4, 1.0), (0.7, 1.0)]


## 1. Connect to GitHub
Use the same hidden GitHub token prompt as your training launcher. Your token needs read access to this repository. The token is passed temporarily, not written into the clone URL.

This downloads shared Python code and updates this Colab checkout. It does not merge any GitHub branches.


In [ ]:
import sys
import base64
import os
import subprocess
from getpass import getpass

# Pass credentials through a temporary child-process environment, not the URL.
token = getpass("GitHub PAT (hidden; repo scope). Leave empty for a public repo: ")
git_env = os.environ.copy()
if token:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {auth}")
    del auth
try:
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], env=git_env, check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], env=git_env, check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], env=git_env, check=True)
finally:
    del token, git_env
os.chdir(REPO_DIR)
actual = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
assert actual == BRANCH, f"Wrong branch: {actual}"
print("Branch:", actual)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


## 2. Install the packages
This can take a few minutes on a fresh runtime.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


## 3. Connect to Google Drive
As in the training launcher, `CS604_RUNS` should exist in My Drive (or be a shortcut to your shared folder). Each inspection gets its own dated folder, so previous results are kept.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive

drive.mount("/content/drive")
base_out = Path(DRIVE_OUT)
if not base_out.is_dir():
    raise FileNotFoundError(f"{DRIVE_OUT} not found. Add the shared CS604_RUNS folder as a shortcut in My Drive, as in colab_launcher_qq.ipynb.")
out = base_out / "dataset_inspection" / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_UTC")
out.mkdir(parents=True, exist_ok=True)
print("Pictures and tables will be saved to:", out)


## 4. Get the images on Colab's temporary disk
**This is usually the slowest setup step.** The existing downloader fetches the whole dataset and keeps the archives as well as the extracted images. A 1,000-image inspection sample does not make the download smaller.

Check free disk space below before continuing. If images are already extracted, set `DOWNLOAD_DATA = False`. Existing extracted training and validation images are reused automatically. Download and extraction messages appear live; extraction may take time between messages. Do not delete archives while extraction is running.


In [ ]:
import shutil
root = Path(DATA_DIR)
root.mkdir(parents=True, exist_ok=True)
usage = shutil.disk_usage(root)
print(f"Colab disk: {usage.free / 2**30:.1f} GiB free / {usage.total / 2**30:.1f} GiB total")
print("Data folder:", root)
print("This is temporary Colab storage, not Drive.")


In [ ]:
# Stream subprocess output into the notebook, and stop on failure.
def run_live(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    except BaseException:
        if process.poll() is None:
            process.terminate()
        process.wait()
        raise


def split_is_extracted(split):
    listing = root / f"{split}.txt"
    if not listing.is_file():
        return False
    # Lightweight reuse check, not a full dataset audit.
    examples = []
    with listing.open() as handle:
        for line in handle:
            parts = line.split()
            if parts:
                examples.append(parts[0])
            if len(examples) == 20:
                break
    return bool(examples) and all((root / path).is_file() for path in examples)

if not DOWNLOAD_DATA:
    print("Download skipped. Using", root)
elif split_is_extracted('train') and split_is_extracted('val'):
    print("Extracted train and validation images found; skipping download.")
else:
    print("Downloading/extracting the full dataset. This can take a while.")
    run_live([sys.executable, "-u", "scripts/download_data.py", "--dest", DATA_DIR])


## 5. Load image lists and choose examples
This simply reads image paths and available labels for the pictures below. It does not audit the lists or create a new split. Training class IDs are numbers; class names are not supplied here.


In [ ]:
import json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance
from tqdm.auto import tqdm
from src.data import ImageListDataset

rng = random.Random(SEED)
frames = []
for split in ['train', 'val']:
    if (root / f'{split}.txt').is_file() or (root / split).is_dir():
        ds = ImageListDataset(root, split)
        frames.append(pd.DataFrame([(split,p,y) for p,y in ds.samples], columns=['split','path','label']))
if not frames:
    raise FileNotFoundError('No image lists or train/val folders found. Check DATA_DIR after extraction.')
images = pd.concat(frames, ignore_index=True)
print('Images available:', images.groupby('split').size().to_dict())
train = images[images.split.eq('train') & images.label.ge(0)].copy()
sampled = pd.concat([g.sample(min(len(g), SAMPLE_PER_SPLIT), random_state=SEED) for _,g in images.groupby('split')], ignore_index=True)

def read_image(path):
    # Matches the trainer: PIL RGB conversion, without changing EXIF orientation.
    with Image.open(root / path) as image:
        return image.convert('RGB')

def save_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(out / filename, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

settings = {key: globals()[key] for key in ['BRANCH','DATA_DIR','CONFIG','SEED','SAMPLE_PER_SPLIT','CLASS_IDS','EXAMPLE_PATHS','EXAMPLE_COUNT','RESOLUTIONS','COMPARE_SIZE','ZOOM_KEEP','ZOOM_CENTER','CROP_RANGES']}
settings['commit'] = subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
(out / 'settings.json').write_text(json.dumps(settings,indent=2))


## 6. Compare training and validation images
Start with size, shape, brightness and color. Strong differences suggest settings worth testing; they do not by themselves explain an accuracy gap. A larger requested resolution cannot restore detail missing from a tiny original.


In [ ]:
records = []
for row in tqdm(sampled.itertuples(index=False), total=len(sampled), desc='Reading inspection sample'):
    try:
        image = read_image(row.path)
        small = np.asarray(image.resize((64,64)), dtype=float) / 255
        records.append(dict(split=row.split,path=row.path,label=row.label,width=image.width,height=image.height,
                            short_side=min(image.size),aspect=image.width/image.height,
                            brightness=np.asarray(image.convert('L').resize((64,64)),dtype=float).mean()/255,
                            red=small[:,:,0].mean(),green=small[:,:,1].mean(),blue=small[:,:,2].mean()))
    except Exception as exc:
        print('Skipped unreadable image:', row.path, type(exc).__name__)
stats = pd.DataFrame(records)
if stats.empty:
    raise RuntimeError('No sampled images could be opened. Check DATA_DIR and extraction.')
stats.to_csv(out / 'sample_image_stats.csv',index=False)
display(stats.groupby('split')[['width','height','short_side','aspect','brightness','red','green','blue']].median().round(3))
fig, axes = plt.subplots(1,3,figsize=(15,4))
for ax, field, title in zip(axes,['short_side','aspect','brightness'],['Original shorter side (pixels)','Shape: width / height','Brightness (0–1)']):
    edges = np.histogram_bin_edges(stats[field],bins=35)
    for split,group in stats.groupby('split'):
        ax.hist(group[field],bins=edges,density=True,alpha=.4,label=split)
    ax.set_title(title); ax.legend()
save_figure(fig,'train_val_sizes_shapes_brightness.png')

def gallery(rows, filename, columns=5):
    rows = list(rows)
    if not rows: return
    fig, axes = plt.subplots((len(rows)+columns-1)//columns,columns,figsize=(columns*3,3*((len(rows)+columns-1)//columns)),squeeze=False)
    for ax in axes.flat: ax.axis('off')
    for ax,row in zip(axes.flat,rows):
        image = read_image(row['path'])
        ax.imshow(image)
        label = f"class {row['label']}" if row['label'] >= 0 else 'unlabeled'
        ax.set_title(f"{row['split']} | {label} | {image.width}×{image.height}\n{Path(row['path']).name}",fontsize=8)
    save_figure(fig,filename)

for split,group in stats.groupby('split'):
    gallery(group.sample(min(10,len(group)),random_state=SEED).to_dict('records'), f'{split}_examples.png')
classes = CLASS_IDS or rng.sample(sorted(train.label.unique().tolist()), min(4,train.label.nunique()))
for class_id in classes:
    group = train[train.label.eq(class_id)]
    gallery(group.sample(min(10,len(group)),random_state=SEED).to_dict('records'), f'class_{class_id}.png')
# Pick a mix of random, small, tall and wide images from both splits.
candidates = []
for _,group in stats.groupby('split'):
    candidates.append(group.sample(1,random_state=SEED))
for field in ['short_side','aspect']:
    candidates.extend([stats.nsmallest(1,field),stats.nlargest(1,field)])
candidates.append(stats.sample(frac=1,random_state=SEED))
auto = pd.concat(candidates).drop_duplicates('path').head(EXAMPLE_COUNT)
example_paths = EXAMPLE_PATHS or auto.path.tolist()
if not example_paths: raise ValueError('Choose at least one example image.')
pd.DataFrame({'path':example_paths}).to_csv(out/'comparison_examples.csv',index=False)
print('Examples used in every comparison:')
for path in example_paths: print(path)


## 7. Original versus current training crops
These are the actual transforms used by the shared trainer for `CONFIG`. Each image gets four random attempts. Watch for a crop that removes the object or the features that distinguish its class. The printed settings show what you already use.


In [ ]:
import torch
from timm.data import resolve_data_config
from timm.models import get_pretrained_cfg
from src.data import build_transforms
from src.utils import load_config, set_seed
cfg = load_config(CONFIG)
train_tf, eval_tf = build_transforms(cfg)
print('Current input size:',cfg.img_size)
print('Current training changes:',train_tf)
print('Current prediction changes:',eval_tf)
(out/'current_transforms.txt').write_text(f'{CONFIG}\nTrain: {train_tf}\nEval: {eval_tf}\n')
pc = get_pretrained_cfg(cfg.model_name)
dc = resolve_data_config({'input_size':(3,cfg.img_size,cfg.img_size)},pretrained_cfg=pc.to_dict() if hasattr(pc,'to_dict') else pc)
mean = torch.tensor(dc['mean'])[:,None,None]
std = torch.tensor(dc['std'])[:,None,None]
def show_tensor(tensor): return (tensor.cpu()*std+mean).clamp(0,1).permute(1,2,0).numpy()
set_seed(SEED)
for i,path in enumerate(example_paths):
    image = read_image(path)
    fig,axes = plt.subplots(1,5,figsize=(16,3.5))
    axes[0].imshow(image); axes[0].set_title('Original')
    for j,ax in enumerate(axes[1:]):
        ax.imshow(show_tensor(train_tf(image))); ax.set_title(f'Current train crop {j+1}')
    for ax in axes: ax.axis('off')
    fig.suptitle(Path(path).name)
    save_figure(fig,f'current_training_{i:02d}.png')


## 8. Compare gentler and stronger random crops
Here only the crop changes. Each row uses a different allowed crop area; each has three attempts. A minimum of 0.7 keeps at least 70% of the original area before resizing, while 0.08 allows a much tighter crop. These are candidate training settings, not measured accuracy results. All outputs have the same size.


In [ ]:
from torchvision.transforms import RandomResizedCrop, InterpolationMode
assert COMPARE_SIZE > 0
for low,high in CROP_RANGES:
    assert 0 < low <= high <= 1
for i,path in enumerate(example_paths):
    image = read_image(path)
    fig,axes = plt.subplots(len(CROP_RANGES),4,figsize=(13,3*len(CROP_RANGES)),squeeze=False)
    for row,(low,high) in enumerate(CROP_RANGES):
        set_seed(SEED+i)  # Repeatable random draws for each setting.
        tf = RandomResizedCrop(COMPARE_SIZE,scale=(low,high),ratio=(.75,4/3),interpolation=InterpolationMode.BICUBIC,antialias=True)
        axes[row,0].imshow(image); axes[row,0].set_title(f'Original | keep {low:.0%}–{high:.0%} area')
        for j in range(1,4):
            axes[row,j].imshow(tf(image)); axes[row,j].set_title(f'Attempt {j}')
    for ax in axes.flat: ax.axis('off')
    fig.suptitle(Path(path).name)
    save_figure(fig,f'crop_strength_{i:02d}.png')


## 9. Compare resolutions: 224 through 512
Every size uses exactly the **same square field of view**. Top: the entire view. Bottom: the middle quarter of that view, enlarged equally so you can compare small details. Set `ZOOM_CENTER` to inspect an off-center object. No model is run at these sizes.

The center square removes the edges of tall/wide originals; we show its boundary on the original first. Larger sizes cost more time and memory in training. Upscaling a tiny source does not create real detail.


In [ ]:
from PIL import ImageDraw
assert RESOLUTIONS and all(isinstance(size,int) and size>0 and size%16==0 for size in RESOLUTIONS), 'Use positive multiples of 16 for ViT-B/16.'
assert len(ZOOM_CENTER)==2 and all(0<=x<=1 for x in ZOOM_CENTER)

def square_box(image, keep=1.0, center=(0.5, 0.5)):
    side = max(1,round(min(image.size)*keep))
    cx,cy = center[0]*image.width, center[1]*image.height
    left = max(0,min(image.width-side,round(cx-side/2)))
    top = max(0,min(image.height-side,round(cy-side/2)))
    return left,top,left+side,top+side

def resize_view(image,size):
    return image.resize((size,size),Image.Resampling.BICUBIC)

for i,path in enumerate(example_paths):
    image = read_image(path)
    box = square_box(image)
    base = image.crop(box)
    marked = image.copy(); ImageDraw.Draw(marked).rectangle(box,outline='red',width=max(2,min(image.size)//100))
    fig,axes = plt.subplots(2,len(RESOLUTIONS)+1,figsize=(3*(len(RESOLUTIONS)+1),6),squeeze=False)
    axes[0,0].imshow(marked); axes[0,0].set_title('Original + square boundary')
    detail_box = square_box(base,.25,ZOOM_CENTER)
    axes[1,0].imshow(base.crop(detail_box),interpolation='nearest'); axes[1,0].set_title('Source detail (25% side)')
    for j,size in enumerate(RESOLUTIONS,1):
        resized = resize_view(base,size)
        axes[0,j].imshow(resized,interpolation='nearest'); axes[0,j].set_title(f'{size} × {size}')
        axes[1,j].imshow(resized.crop(square_box(resized,.25,ZOOM_CENTER)),interpolation='nearest'); axes[1,j].set_title('Same detail, enlarged')
    for ax in axes.flat: ax.axis('off')
    fig.suptitle(f'{Path(path).name} | original {image.width} × {image.height}')
    save_figure(fig,f'resolution_{i:02d}.png')


## 10. Compare simple zoom levels
The red boxes show what will be kept. Each close-up has the **same output size**, so this changes framing, not resolution. A 0.65 setting keeps 65% of the base square's side (about 42% of its area).

These are simple crops, not the model-guided zoom used during prediction. Look for the point where useful details become clearer, and where parts of the object or helpful surroundings disappear. `ZOOM_CENTER` chooses the location; it does not detect an object.


In [ ]:
assert ZOOM_KEEP and all(0<keep<=1 for keep in ZOOM_KEEP)
for i,path in enumerate(example_paths):
    image = read_image(path)
    base = image.crop(square_box(image))
    fig,axes = plt.subplots(2,len(ZOOM_KEEP),figsize=(4*len(ZOOM_KEEP),7),squeeze=False)
    for col,keep in enumerate(ZOOM_KEEP):
        box = square_box(base,keep,ZOOM_CENTER)
        marked = base.copy(); ImageDraw.Draw(marked).rectangle(box,outline='red',width=max(2,min(base.size)//100))
        axes[0,col].imshow(marked); axes[0,col].set_title(f'Keep {keep:.0%} of square side')
        axes[1,col].imshow(resize_view(base.crop(box),COMPARE_SIZE)); axes[1,col].set_title(f'Close-up → {COMPARE_SIZE} × {COMPARE_SIZE}')
    for ax in axes.flat: ax.axis('off')
    fig.suptitle(Path(path).name)
    save_figure(fig,f'zoom_{i:02d}.png')


## 11. Flip and mild brightness/color changes
Each change starts from the same square image; changes are not stacked. Check whether text becomes unreadable, colors that identify a class are altered, or the result looks unrealistic. These pictures are options to review, not a recommendation to apply every change.


In [ ]:
from PIL import ImageOps
for i,path in enumerate(example_paths):
    image = read_image(path)
    base = resize_view(image.crop(square_box(image)),COMPARE_SIZE)
    variants = [('Original square',base),('Horizontal flip',ImageOps.mirror(base)),
                ('Slightly darker (0.8×)',ImageEnhance.Brightness(base).enhance(.8)),
                ('Slightly brighter (1.2×)',ImageEnhance.Brightness(base).enhance(1.2)),
                ('Less color (0.8×)',ImageEnhance.Color(base).enhance(.8)),
                ('More color (1.2×)',ImageEnhance.Color(base).enhance(1.2))]
    fig,axes = plt.subplots(2,3,figsize=(12,8))
    for ax,(title,view) in zip(axes.flat,variants):
        ax.imshow(view); ax.set_title(title); ax.axis('off')
    fig.suptitle(Path(path).name)
    save_figure(fig,f'flip_brightness_color_{i:02d}.png')


## 12. Decide what to try in training
After reviewing the pictures, write down what you saw:

- **Cropping:** which settings keep the object reliably over several attempts?
- **Resolution:** what is the smallest size that keeps the details you care about?
- **Zoom:** where does a closer view help, and where does it cut too much away?
- **Flip and color:** which changes still look like valid examples of the same class?
- **Train versus validation:** do framing, shapes or lighting look different?

Choose two or three promising settings and compare them in small training runs. Keep the same evaluation images and budget, changing one thing at a time. Keep evaluation images out of training and prototype building. A few pictures cannot establish what works for all 10,000 classes: use `CLASS_IDS`, `EXAMPLE_PATHS` and a different seed to review more examples.

All pictures, settings and the sample table are in the Drive folder printed below. You can share selected pictures here to discuss the next experiment.


In [ ]:
review = """What did I see?
Crop settings that keep useful parts:
Smallest useful resolution:
Zoom levels that help / cut too much:
Flip or color changes to avoid:
Training / validation differences:
Two or three settings to test next:
"""
(out/'review_notes.txt').write_text(review)
print('Finished. Results saved to:',out)
print('Edit review_notes.txt in Drive to record your observations.')
